In [1]:
import pandas as pd

WDI_CODES = [
    "NV.IND.MANF.ZS", "NV.IND.TOTL.ZS", "NV.AGR.TOTL.ZS", "NV.SRV.TOTL.ZS",
    "SL.IND.EMPL.ZS", "SL.AGR.EMPL.ZS", "SL.SRV.EMPL.ZS", "SL.TLF.CACT.ZS",
    "SL.TLF.CACT.FE.ZS", "SL.EMP.SELF.ZS", "SL.EMP.VULN.ZS", "SL.UEM.TOTL.ZS",
    "SL.EMP.TOTL.SP.ZS", "SL.TLF.TOTL.IN", "SP.URB.TOTL.IN.ZS", "NE.TRD.GNFS.ZS",
    "NE.EXP.GNFS.ZS", "BX.KLT.DINV.WD.GD.ZS", "TX.VAL.MANF.ZS.UN", "NY.GDP.PCAP.KD",
    "NY.GDP.MKTP.KD.ZG", "FP.CPI.TOTL.ZG", "GC.TAX.TOTL.GD.ZS", "NE.CON.GOVT.ZS",
    "SE.XPD.TOTL.GD.ZS", "SH.XPD.GHED.GD.ZS", "SP.POP.DPND", "SE.SEC.CUAT.UP.ZS",
    "SE.TER.ENRR", "SM.POP.TOTL.ZS", "SI.POV.GINI",
]

WDI_NAMES = {
    "NV.IND.MANF.ZS":       "manufacturing_va_pct_gdp",
    "NV.IND.TOTL.ZS":       "industry_va_pct_gdp",
    "NV.AGR.TOTL.ZS":       "agriculture_va_pct_gdp",
    "NV.SRV.TOTL.ZS":       "services_va_pct_gdp",
    "SL.IND.EMPL.ZS":       "emp_industry_pct",
    "SL.AGR.EMPL.ZS":       "emp_agriculture_pct",
    "SL.SRV.EMPL.ZS":       "emp_services_pct",
    "SL.TLF.CACT.ZS":       "lfp_rate",
    "SL.TLF.CACT.FE.ZS":    "lfp_rate_female",
    "SL.EMP.SELF.ZS":       "self_employed_pct",
    "SL.EMP.VULN.ZS":       "vulnerable_emp_pct",
    "SL.UEM.TOTL.ZS":       "unemployment_rate",
    "SL.EMP.TOTL.SP.ZS":    "emp_pop_ratio",
    "SL.TLF.TOTL.IN":       "labor_force_total",
    "SP.URB.TOTL.IN.ZS":    "urban_pop_pct",
    "NE.TRD.GNFS.ZS":       "trade_pct_gdp",
    "NE.EXP.GNFS.ZS":       "exports_pct_gdp",
    "BX.KLT.DINV.WD.GD.ZS": "fdi_inflows_pct_gdp",
    "TX.VAL.MANF.ZS.UN":    "manuf_exports_pct",
    "NY.GDP.PCAP.KD":       "gdp_pc_const",
    "NY.GDP.MKTP.KD.ZG":    "gdp_growth",
    "FP.CPI.TOTL.ZG":       "inflation_cpi",
    "GC.TAX.TOTL.GD.ZS":    "tax_revenue_pct_gdp",
    "NE.CON.GOVT.ZS":       "govt_consumption_pct_gdp",
    "SE.XPD.TOTL.GD.ZS":    "educ_spend_pct_gdp",
    "SH.XPD.GHED.GD.ZS":    "health_spend_pct_gdp",
    "SP.POP.DPND":          "age_dependency_ratio",
    "SE.SEC.CUAT.UP.ZS":    "upper_secondary_attain_pct",
    "SE.TER.ENRR":          "tertiary_enroll_gross",
    "SM.POP.TOTL.ZS":       "migrant_stock_pct",
    "SI.POV.GINI":          "gini",
}

# load and filter to the indicators of interest
raw = pd.read_csv("../data/WDICSV.csv")
raw = raw[raw["Indicator Code"].isin(WDI_CODES)]

missing = sorted(set(WDI_CODES) - set(raw["Indicator Code"]))
if missing:
    print(f"{len(missing)} codes not in this WDI vintage: {missing}")

# reshape to one row per country-year
year_cols = [c for c in raw.columns if c.isdigit()]
panel = (
    raw.set_index(["Country Code", "Country Name", "Indicator Code"])[year_cols]
       .rename_axis(columns="year")
       .stack()
       .unstack("Indicator Code")
       .reset_index()
       .rename_axis(columns=None)
)
panel["year"] = panel["year"].astype(int)

# drop regional and income aggregates
meta = pd.read_csv("../data/WDICountry.csv")
panel = panel[panel["Country Code"].isin(meta.loc[meta["Region"].notna(), "Country Code"])]

# order columns and rename
panel = (
    panel[["Country Code", "Country Name", "year"] + [c for c in WDI_CODES if c in panel.columns]]
    .rename(columns={**WDI_NAMES, "Country Code": "iso3", "Country Name": "country"})
    .sort_values(["iso3", "year"])
    .reset_index(drop=True)
)

In [2]:
# fill gaps within each country: straight line between observed years, and carry
# the first/last value up to 3 years past the ends. runs on all years (not just
# 2000+) so earlier observations can anchor the fill
value_cols = [c for c in panel.columns if c not in ["iso3", "country", "year"]]

filled = panel.copy()
filled[value_cols] = (
    panel.groupby("iso3")[value_cols]
         .transform(lambda s: s.interpolate(limit_area="inside").ffill(limit=3).bfill(limit=3))
)

wb = filled[filled["year"] >= 2000]

# % of country-years missing, before vs after the fill
print(
    pd.DataFrame({
        "before": panel.loc[panel["year"] >= 2000, value_cols].isna().mean(),
        "after":  wb[value_cols].isna().mean(),
    })
    .mul(100).round(1)
    .sort_values("before", ascending=False)
    .head(10)
)

                            before  after
migrant_stock_pct             77.0    0.5
gini                          67.0   32.7
upper_secondary_attain_pct    61.6   22.6
tax_revenue_pct_gdp           46.5   36.0
tertiary_enroll_gross         43.5   21.1
educ_spend_pct_gdp            40.3   13.9
manuf_exports_pct             25.8   14.9
govt_consumption_pct_gdp      21.8   18.2
health_spend_pct_gdp          19.4   12.2
exports_pct_gdp               18.7   15.3


In [3]:
union = pd.read_csv("../data/ILR_TUMT_NOC_RT_A-full-2026-09-16.csv")

import country_converter as coco

# convert each unique name once, then map it back to every row
names = union["ref_area.label"].unique()
union["iso3"] = union["ref_area.label"].map(dict(zip(names, coco.convert(names, to="ISO3"))))

union = union.rename(columns={"time": "year", "obs_value": "union_density"})

union_raw = union.drop(columns=["note_indicator.label","note_source.label","source.label","indicator.label"])
union = union_raw.rename(columns = {"obs_value":"union_density"})

print(union.head())

  ref_area.label  year  union_density iso3
0    Afghanistan  2019         16.767  AFG
1    Afghanistan  2012          9.362  AFG
2        Albania  2017         36.847  ALB
3        Albania  2013         48.698  ALB
4        Albania  2012         48.204  ALB


In [4]:
vdem = pd.read_csv("../data/V-Dem-CY-Full+Others-v16.csv")


C:\Users\Joshu\AppData\Local\Temp\ipykernel_28804\3164604489.py:1: DtypeWarning: Columns (0: v2lpname, 1: v2slpname, 2: v2tlpname, 3: v2elregnam, 4: v2ellocnam, 5: v2exparhos, 6: v2exnamhog, 7: v2extithog, 8: v2expothog, 9: v2lgnameup, 10: v2lgnamelo, 11: v2juhcname, 12: v3lgcamoth, 13: v3lgnamelo, 14: v3lgnameup, 15: v3exnamhos, 16: v3exnamhog, 17: v3extithos, 18: v3extithog, 19: v3elregnam, 20: v3ellocnam, 21: v3juhcname) have mixed types. Specify dtype option on import or set low_memory=False.
  vdem = pd.read_csv("../data/V-Dem-CY-Full+Others-v16.csv")


In [5]:

vdem = vdem[vdem["year"]>= 2000]

VDEM_NAMES = {
    "v2x_frassoc_thick": "freedom_of_association",
    "v2xcs_ccsi":        "civil_society_index",
    "v2x_polyarchy":     "electoral_democracy",
    "v2x_libdem":        "liberal_democracy",
    "v2csreprss":        "cso_repression",
    "v2cseeorgs":        "cso_entry_exit",
    "v2x_egal":          "egalitarian_component",
    "v2xeg_eqdr":        "equal_resource_distribution",
    "v2pepwrses":        "power_by_socioeconomic_position",
}

missing = sorted(set(VDEM_NAMES) - set(vdem.columns))
if missing:
    print(f"{len(missing)} not found: {missing}")

vdem = (
    vdem[["country_text_id", "year"] + [c for c in VDEM_NAMES if c in vdem.columns]]
    .rename(columns={**VDEM_NAMES, "country_text_id": "iso3"})
    .sort_values(["iso3", "year"])
    .reset_index(drop=True)
)

In [6]:
df = (
    union[["iso3", "year", "union_density"]]
    .merge(wb,   left_on=["iso3", "year"], right_on=["iso3", "year"], how="left")
    .merge(vdem, left_on=["iso3", "year"], right_on=["iso3", "year"], how="left")
)

df.head(10)

,iso3,year,union_density,country,manufacturing_va_pct_gdp,industry_va_pct_gdp,agriculture_va_pct_gdp,services_va_pct_gdp,emp_industry_pct,emp_agriculture_pct,...,gini,freedom_of_association,civil_society_index,electoral_democracy,liberal_democracy,cso_repression,cso_entry_exit,egalitarian_component,equal_resource_distribution,power_by_socioeconomic_position
0,AFG,2019,16.767,Afghanistan,7.043181,14.058112,25.773971,55.472807,18.609292,43.877169,...,NaN,0.593,0.617,0.329,0.162,0.863,0.586,0.393,0.290,-0.834
1,AFG,2012,9.362,Afghanistan,12.486085,21.157807,24.390874,50.579398,18.922538,48.764168,...,NaN,0.638,0.694,0.350,0.194,1.563,0.722,0.453,0.320,-0.565
2,ALB,2017,36.847,Albania,6.516401,22.953333,18.079078,44.562131,19.946067,37.811593,...,33.10,0.818,0.806,0.536,0.437,1.676,1.508,0.655,0.723,-0.196
3,ALB,2013,48.698,Albania,5.747654,25.506804,18.924817,42.628784,17.322531,43.961675,...,31.80,0.815,0.811,0.523,0.433,1.676,1.508,0.677,0.743,-0.196
4,ALB,2012,48.204,Albania,5.379378,25.407724,17.591369,43.505924,17.032465,46.130692,...,29.00,0.838,0.859,0.540,0.444,1.622,1.784,0.665,0.743,-0.075
5,ALB,2011,49.900,Albania,6.192222,26.940987,16.720223,42.239020,18.771647,45.440017,...,29.25,0.840,0.862,0.539,0.441,2.074,1.608,0.665,0.743,-0.075
6,ALB,2009,49.496,Albania,5.656709,27.567872,15.279222,42.004024,18.724676,44.720796,...,29.75,0.838,0.862,0.544,0.446,2.074,1.608,0.657,0.706,-0.075
7,ALB,2008,52.406,Albania,5.422753,28.561650,15.422100,41.075029,17.832877,44.626125,...,30.00,0.837,0.862,0.551,0.447,2.074,1.608,0.655,0.702,-0.075
8,ARG,2014,27.700,Argentina,14.772779,24.277394,6.712704,52.940543,22.794876,8.174670,...,41.80,0.870,0.832,0.777,0.595,1.313,1.442,0.768,0.700,1.753
9,ARG,2013,30.400,Argentina,15.011054,24.005881,6.052918,53.919094,22.474095,8.227696,...,41.10,0.869,0.832,0.772,0.588,1.313,1.442,0.768,0.700,1.753


In [7]:
df.to_csv("../data/merged.csv")